# Traditional ML Workflow: Fraud Detection

A pure open-source approach using pandas, scikit-learn, and XGBoost - no Snowflake dependencies.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

## Step 1: Load Data

In [2]:
df = pd.read_csv('data/transactions.csv')
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (1754155, 9)


,TRANSACTION_ID,TX_DATETIME,CUSTOMER_ID,TERMINAL_ID,TX_AMOUNT,TX_TIME_SECONDS,TX_TIME_DAYS,TX_FRAUD,TX_FRAUD_SCENARIO
0,0,2018-04-01 00:00:31,596,3156,57.16,31,0,0,0
1,1,2018-04-01 00:02:10,4961,3412,81.51,130,0,0,0
2,2,2018-04-01 00:07:56,2,1365,146.00,476,0,0,0
3,3,2018-04-01 00:09:29,4128,8737,64.49,569,0,0,0
4,4,2018-04-01 00:10:34,927,9906,50.99,634,0,0,0


## Step 2: Feature Engineering

In [3]:
df['TX_DATETIME'] = pd.to_datetime(df['TX_DATETIME'])

df['HOUR'] = df['TX_DATETIME'].dt.hour
df['DAY_OF_WEEK'] = df['TX_DATETIME'].dt.dayofweek
df['IS_WEEKEND'] = df['DAY_OF_WEEK'] >= 5
df['IS_NIGHT_12AM_7AM'] = df['HOUR'].between(0, 6)

print("Features created:")
print(df[['TX_DATETIME', 'HOUR', 'DAY_OF_WEEK', 'IS_WEEKEND', 'IS_NIGHT_12AM_7AM']].head())

Features created:
          TX_DATETIME  HOUR  DAY_OF_WEEK  IS_WEEKEND  IS_NIGHT_12AM_7AM
0 2018-04-01 00:00:31     0            6        True               True
1 2018-04-01 00:02:10     0            6        True               True
2 2018-04-01 00:07:56     0            6        True               True
3 2018-04-01 00:09:29     0            6        True               True
4 2018-04-01 00:10:34     0            6        True               True


## Step 3: Prepare Train/Test Split

In [4]:
basic_features = ['TX_AMOUNT', 'HOUR', 'DAY_OF_WEEK', 'IS_WEEKEND', 'IS_NIGHT_12AM_7AM']
X = df[basic_features].astype(float)
y = df['TX_FRAUD'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Features: {basic_features}")
print(f"\nClass distribution (train): {y_train.value_counts().to_dict()}")

Training set: 1403324 samples
Test set: 350831 samples
Features: ['TX_AMOUNT', 'HOUR', 'DAY_OF_WEEK', 'IS_WEEKEND', 'IS_NIGHT_12AM_7AM']

Class distribution (train): {0: 1391579, 1: 11745}


## Step 4: Train XGBoost Model

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2]
}

model = GradientBoostingClassifier(random_state=42)

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

model = grid_search.best_estimator_

print(f"\n=== GridSearchCV Results ===")
print(f"Best params: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")
print("Model trained successfully")

Model trained successfully


## Step 5: Model Predictions on Test Data

In [6]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1_score": f1_score(y_test, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_proba)
}

print("=== Model Results ===")
for metric, value in metrics.items():
    print(f"{metric}: {value:.4f}")

=== Model Results ===
accuracy: 0.9934
precision: 0.9374
recall: 0.2245
f1_score: 0.3622
roc_auc: 0.6536


In [7]:
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['Not Fraud', 'Fraud']))

print("\n=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))


=== Classification Report ===
              precision    recall  f1-score   support

   Not Fraud       0.99      1.00      1.00    347895
       Fraud       0.94      0.22      0.36      2936

    accuracy                           0.99    350831
   macro avg       0.97      0.61      0.68    350831
weighted avg       0.99      0.99      0.99    350831


=== Confusion Matrix ===
[[347851     44]
 [  2277    659]]


## Step 6: Feature Importance

In [8]:
importance_df = pd.DataFrame({
    'feature': basic_features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("=== Feature Importance ===")
print(importance_df.to_string(index=False))

=== Feature Importance ===
          feature  importance
        TX_AMOUNT    0.958621
             HOUR    0.020097
      DAY_OF_WEEK    0.016378
       IS_WEEKEND    0.004110
IS_NIGHT_12AM_7AM    0.000795
